Checking PSD of R for DINO v3 on COCO dataset

PSD of R <=> eigenvalues of R >= 0

In [ ]:
# download images
!mkdir -p coco
%cd coco
!wget http://images.cocodataset.org/zips/val2017.zip
!unzip -q val2017.zip
%cd ..

In [ ]:
!pip install torchmetrics -q

In [ ]:
import os
import torch
import torch.nn.functional as F
from torchvision import transforms
import torchmetrics
import cv2

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
def make_transform(resize_size = 768):
    #to_tensor = transforms.ToTensor()
    resize = transforms.Resize((resize_size, resize_size), antialias=True)
    normalize = transforms.Normalize(
        mean=(0.485, 0.456, 0.406),
        std=(0.229, 0.224, 0.225),
    )
    return transforms.Compose([resize, normalize])

In [ ]:
img_size = 768
transform = make_transform(img_size)

In [ ]:
!git clone https://github.com/facebookresearch/dinov3.git

In [ ]:
!cp dinov3/hubconf.py .

In [ ]:
import sys
sys.path.append("dinov3")

In [ ]:
!wget http://www.agentspace.org/download/mydinov3.zip
!unzip -P dino mydinov3.zip

In [ ]:
backbone = torch.hub.load('.', 'dinov3_vits16', source='local', weights='dinov3_vits16_pretrain_lvd1689m.pth') # dinov3_vits16

In [ ]:
backbone.eval()

In [ ]:
import inspect
print(inspect.getsource(backbone.__class__))

In [ ]:
test_dir = "coco/val2017"
files = os.listdir(test_dir)
len(files)

In [ ]:
features = []
mn = 1e9
backbone.to(device).to(torch.bfloat16)
for i, f in enumerate(files):
    img = cv2.imread(os.path.join(test_dir, f))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    blob = cv2.dnn.blobFromImage(img,1.0/255)
    with torch.inference_mode():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            inp = transform(torch.tensor(blob[0])).unsqueeze(0).to(device)
            #out = backbone.forward_features(inp)
            out = backbone.get_intermediate_layers(inp,n=[11],norm=False)[0]

    #feats = out['x_prenorm'][0][1+backbone.n_storage_tokens:].view(48,48,-1)
    feats = out[0].view(48,48,-1)
    features.append(feats)
    flat = feats.view(-1, feats.shape[-1])  # shape: (H*W, C)
    W = flat @ flat.T
    minimum = W.min().item()
    if minimum < mn:
        mn = minimum
        print(i,minimum, "!!!!!!" if minimum <= 0 else "")

In [ ]:
features = torch.stack(features)

In [ ]:
features.shape

In [ ]:
features.min(),features.max()

The values of W are always positive, d is well-defined, and R is PSD for all samples